# 工具类型

学习目标：能选择内置工具类型表达更新、筛选和函数关系，并识别浅层处理、推断来源及 this 上下文的边界。

前置知识：泛型、keyof、映射与条件类型、函数和构造签名、Promise、this 参数。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict；另启用 exactOptionalPropertyTypes。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/16-utility-types/。

1. [main.ts](scripts/16-utility-types/main.ts)：配套实现与示例。
2. [tsconfig.json](scripts/16-utility-types/tsconfig.json)：本章独立项目配置。
3. [type-errors.ts](scripts/16-utility-types/type-errors.ts)、[tsconfig.errors.json](scripts/16-utility-types/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:16
```

Step 2：生成本章 JavaScript。

```bash
npm run build:16
```

Step 3：运行本章正常示例。

```bash
npm run run:16
# 正常退出；各段预期输出见代码注释。
```

正常片段均按正文顺序节选自 main.ts，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 对象属性工具与浅层范围

工具类型是在类型层复用常见变换的名称；它们可直接使用，不需要运行时导入。下表 T 表示输入类型，K 表示键的集合。

| 完整名称 | 中文名称／含义 | 作用 |
| --- | --- | --- |
| Partial\<T\> | 部分属性类型 | 全部直接属性变可选 |
| Required\<T\> | 必需属性类型 | 移除直接属性的可选标记 |
| Readonly\<T\> | 只读属性类型 | 限制直接属性重新赋值 |
| Record\<K, T\> | 键值记录类型 | 给 K 中每个键指定值类型 T |
| Pick\<T, K\> | 选取属性类型 | 保留指定键 |
| Omit\<T, K\> | 省略属性类型 | 去掉指定键 |

这些工具不会深层递归变换对象。Partial 允许省略 profile，却不允许在提供 profile 时漏掉它内部的必需字段；Readonly 也不会冻结内部对象。

```typescript
export type Account = { id: number; name: string; profile: { city: string } };
const patch: Partial<Account> = { name: "阿青" };
const required: Required<{ name?: string }> = { name: "阿青" };
const readonlyView: Readonly<Account> = { id: 1, name: "阿青", profile: { city: "杭州" } };
readonlyView.profile.city = "上海";
const counts: Record<"done" | "todo", number> = { done: 2, todo: 3 };
const summary: Pick<Account, "id" | "name"> = { id: 1, name: "阿青" };
const newAccount: Omit<Account, "id"> = { name: "小林", profile: { city: "苏州" } };
console.log(patch.name, required.name, readonlyView.profile.city, counts.done, summary.id, newAccount.name);
// 预期输出：阿青 阿青 上海 2 1 小林
```

## 2 组合更新类型与阅读定义

先用与 Pick 相同的结构定义 Select，具体读一次工具声明。T 表示源类型，K extends keyof T 要求所选键属于 T 的键联合；这里是泛型约束，不是条件类型分支。[P in K] 在类型层逐个遍历 K 中的键，P 表示当前键；T[P] 取出该键在源类型中的属性类型。

```typescript
type Select<T, K extends keyof T> = {
  [P in K]: T[P];
};
type AccountSummary = Select<Account, "id" | "name">; // 等价于 { id: number; name: string }。
```

代入现有 Account 后，K 为 "id" | "name"，符合 keyof Account 的约束。P 为 "id" 时，T[P] 是 number；P 为 "name" 时，T[P] 是 string，因此 AccountSummary 保留这两个属性及各自类型。这些声明只构造类型，不读取或筛选运行时对象。

组合更新类型时再明确需求：更新请求必须有 id，允许修改 name 和 profile，不允许把 id 当成可缺省属性。Pick 保留身份，Omit 去掉身份后再 Partial，最后交叉合并。

Omit 在类型中去掉属性，并不会从已有运行时对象删除字段。结构赋值仍可能接受带有额外字段的变量，不能把它当数据脱敏函数。

```typescript
type Update = Pick<Account, "id"> & Partial<Omit<Account, "id">>;
const update: Update = { id: 1, name: "新名字" };
const stored: Account = { id: 2, name: "保留对象", profile: { city: "南京" } };
const visible: Omit<Account, "id"> = stored;
console.log(update.id, "id" in visible);
// 预期输出：1 true
```

## 3 联合筛选工具

这里 T 是待筛选的联合，U 是用于比较的类型。Exclude 移除能赋给 U 的成员，Extract 保留这些成员；它们处理联合成员，不是对象的属性名。NonNullable 去掉 null 和 undefined，并不检查当前值是否为空。

| 完整名称 | 中文名称／含义 |
| --- | --- |
| Exclude\<T, U\> | 排除符合 U 的联合成员 |
| Extract\<T, U\> | 提取符合 U 的联合成员 |
| NonNullable\<T\> | 排除空值类型 |

```typescript
type State = "draft" | "ready" | "archived";
const active: Exclude<State, "archived"> = "ready";
const finalState: Extract<State, "ready" | "archived"> = "archived";
const text: NonNullable<string | null | undefined> = "存在";
console.log(active, finalState, text);
// 预期输出：ready archived 存在
```

## 4 提取函数与构造签名

函数名是值，先用 typeof 取得类型。Parameters 提取参数元组，ReturnType 提取返回类型；ConstructorParameters 对构造签名提取参数，InstanceType 得到实例类型。T 在这组工具中必须满足相应的调用或构造签名约束。

| 完整名称 | 中文名称／含义 |
| --- | --- |
| Parameters\<T\> | 函数参数元组 |
| ReturnType\<T\> | 函数返回类型 |
| ConstructorParameters\<T\> | 构造参数元组 |
| InstanceType\<T\> | 构造结果的实例类型 |

对重载提取采用最后的可见签名，不能按某组实参自动选择重载。泛型函数的未定类型参数可能得到 unknown，这不同于实际调用时的推断。

```typescript
function format(code: number, label: string) { return `${label}:${code}`; }
const args: Parameters<typeof format> = [3, "条目"];
const formatted: ReturnType<typeof format> = format(...args);
class Ticket {
  constructor(public code: number) {}
}
const constructorArgs: ConstructorParameters<typeof Ticket> = [8];
const ticket: InstanceType<typeof Ticket> = new Ticket(...constructorArgs);
console.log(formatted, ticket.code);
// 预期输出：条目:3 8
```

## 5 Awaited 提取等待后的值

Awaited\<T\> 描述类似 await 的逐层解包关系，也能处理联合中的非 Promise 成员。它是类型工具，不会启动或等待任务；异步函数的真实返回值仍是 Promise。

下面先取得函数的 Promise 返回类型，再用 Awaited 提取记录结构；真正等待由顶层 await 完成，不依赖计时器或网络。

```typescript
async function loadLocal() { return { total: 4 }; }
type Loaded = Awaited<ReturnType<typeof loadLocal>>;
const loaded: Loaded = await loadLocal();
const nested: Awaited<Promise<Promise<number>>> = 6;
console.log(loaded.total, nested);
// 预期输出：4 6
```

## 6 NoInfer 控制推断来源

NoInfer\<T\> 阻止当前位置为 T 提供推断候选，之后仍按 T 检查值。它不是关闭检查，也不阻止显式给出类型实参。

下面 C 代表可选模式，应该只从 choices 推断；fallback 负责满足已有模式集合。如果允许它参与推断，错误的新模式可能扩大 C。本例还用实际包含检查防御未检查的运行时调用。

```typescript
export function choose<C extends string>(choices: readonly C[], fallback: NoInfer<C>): C {
  if (!choices.includes(fallback)) throw new Error("默认值必须在候选中");
  return fallback;
}
console.log(choose(["read", "write"], "read"));
// 预期输出：read
```

## 7 this 参数的提取与移除

ThisParameterType\<T\> 取得显式 this 参数，没有该参数时得到 unknown。OmitThisParameter\<T\> 移除签名里的 this 参数，实际绑定仍须调用 bind；它不会自动保存接收者。对于重载与泛型，其转换还会丢失部分签名信息。

| 完整名称 | 中文名称／含义 |
| --- | --- |
| ThisParameterType\<T\> | 提取显式 this 参数类型 |
| OmitThisParameter\<T\> | 移除显式 this 参数的函数类型 |
| ThisType\<T\> | 指定对象字面量中的上下文 this |

```typescript
function show(this: { label: string }, count: number) { return `${this.label}:${count}`; }
const receiver: ThisParameterType<typeof show> = { label: "记录" };
const bound: OmitThisParameter<typeof show> = show.bind(receiver);
console.log(bound(2));
// 预期输出：记录:2
```

## 8 ThisType 是上下文标记

ThisType 本身不产生新值，也不是一种 this 绑定操作；它为对象字面量的方法提供上下文 this 类型，需要启用 noImplicitThis，本章 strict 已包含这项检查。

下面 CounterState 是计数状态，Methods 是方法结构。方法体可按 CounterState 使用 this；对象展开真正把状态和方法放到同一接收者上，调用方式保证运行时 this 指向该对象。

```typescript
type CounterState = { count: number };
type Methods = { increment(): void };
const methods: Methods & ThisType<CounterState & Methods> = {
  increment() { this.count += 1; }
};
const counter = { count: 0, ...methods };
counter.increment();
console.log(counter.count);
// 预期输出：1
```

## 9 检查类型边界

下面的 [type-errors.ts](scripts/16-utility-types/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import { choose, type Account } from "./main.js";
const shallow: Partial<Account> = { profile: {} }; // profile 出现后仍必须有 city。
const noId: Pick<Account, "id"> & Partial<Omit<Account, "id">> = {}; // id 必需。
const missingKey: Record<"done" | "todo", number> = { done: 1 }; // todo 不能省略。
const nullText: NonNullable<string | null> = null; // 类型已排除 null。
choose(["read", "write"], "delete"); // fallback 不扩大推断出的模式集合。
type NotFunction = ReturnType<string>; // string 不符合调用签名约束。
// 预期诊断包含：TS2741, TS2322, TS2345, TS2344。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:16
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

对象工具、联合工具、签名工具处理的对象不同。组合工具先保留业务所需关系；浅层类型操作不清洗数据、不创建默认值、不执行等待，也不绑定 this。

## 练习

1. 新增只允许更新 name 的请求类型且 id 必需；核对 { id: 1 } 合法，直接字面量中的 profile 被拒绝。

2. 给 format 新增可选后缀参数，观察 Parameters 的元组如何变化；用参数元组展开调用并运行。

3. 把 choose 的第二个参数改成 delete，确认类型拒绝；再从 JavaScript 调用相同实现，核对运行时包含检查仍能拒绝非法值。

4. 把 counter 初值改为 4 并调用两次 increment，核对结果为 6；说明 ThisType 为什么没有负责创建 count。

## 参考与引用来源

- TypeScript 官方文档：[Utility Types：本章各工具的同名小节](https://www.typescriptlang.org/docs/handbook/utility-types.html)；[Mapped Types：键遍历](https://www.typescriptlang.org/docs/handbook/2/mapped-types.html)；[Indexed Access Types：按键取得属性类型](https://www.typescriptlang.org/docs/handbook/2/indexed-access-types.html)；[2.8：Predefined conditional types 与映射修饰符](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-8.html#predefined-conditional-types)；[5.4：The NoInfer Utility Type](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-4.html#the-noinfer-utility-type)；[4.5：The Awaited Type and Promise Improvements](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-5.html#the-awaited-type-and-promise-improvements)；[Object Types：readonly 的浅层边界](https://www.typescriptlang.org/docs/handbook/2/objects.html#readonly-properties)。